In [ ]:
!pip install -U transformers

exemple test

In [ ]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("image-text-to-text", model="LiquidAI/LFM2.5-VL-1.6B")
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/p-blog/candy.JPG"},
            {"type": "text", "text": "What animal is on the candy?"}
        ]
    },
]
pipe(text=messages)

Framing a vid and asking for description img per img (1 per s fps)

In [ ]:
# =========================================
# VIDEO → 1 FRAME / SECOND → VLM (COLAB)
# USING FFMPEG (100% RELIABLE)
# =========================================

# 1️⃣ Install dependencies
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!pip install -q transformers torch pillow accelerate

# 2️⃣ Upload video
from google.colab import files
uploaded = files.upload()

video_name = list(uploaded.keys())[0]
video_path = f"/content/{video_name}"

# 3️⃣ Extract 1 image per second using FFMPEG
import os
frames_dir = "/content/frames"
os.makedirs(frames_dir, exist_ok=True)

!ffmpeg -i "$video_path" -vf fps=1 "$frames_dir/frame_%04d.jpg"

print("✅ Frames extracted with ffmpeg")

# 4️⃣ Load Vision-Language model
from transformers import pipeline

pipe = pipeline(
    "image-text-to-text",
    model="LiquidAI/LFM2.5-VL-1.6B",
    device=0  # GPU
)

# 5️⃣ Describe each frame
image_files = sorted([
    f for f in os.listdir(frames_dir)
    if f.endswith(".jpg")
])

for img_name in image_files:
    img_path = os.path.join(frames_dir, img_name)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "url": img_path},
                {"type": "text", "text": "Décris cette image"}
            ]
        }
    ]

    result = pipe(text=messages)

    print(f"\n🖼️ {img_name}")
    print("🧠 Description :")
    print(result[0]["generated_text"])
    print("-" * 60)
